# T2I-RL Training on Colab Enterprise

**Full training pipeline for Text-to-Image Generation with Reinforcement Learning**

This notebook is optimized for Colab Enterprise with T4 16GB GPU + 50GB RAM.

## What this notebook does:
1. Setup environment and dependencies
2. Load Janus-Pro-1B model (bfloat16)
3. Setup CLIP reward model
4. Run GRPO training
5. Evaluate results

---

## 0. Check GPU

In [ ]:
import torch
import sys

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.1f} GB")
else:
    raise RuntimeError("No GPU available!")

## 1. Install Dependencies

In [ ]:
import time
start = time.time()

# Clean install
!pip uninstall -y transformers accelerate peft 2>/dev/null || true

# Install compatible versions
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.49.0 accelerate safetensors
!pip install -q --no-deps git+https://github.com/deepseek-ai/Janus.git
!pip install -q attrdict sentencepiece
!pip install -q open-clip-torch timm einops peft bitsandbytes
!pip install -q tqdm Pillow matplotlib wandb

print(f"\nInstallation complete: {time.time()-start:.1f}s")
print("\n" + "="*50)
print("RESTART RUNTIME NOW: Runtime -> Restart session")
print("="*50)

---
## After Restart, Continue Here
---

In [ ]:
# Clone and setup project
import os
import sys

if not os.path.exists('T2I-RL-Project'):
    !git clone https://github.com/Inoriany/T2I-RL-Project.git
else:
    %cd T2I-RL-Project
    !git pull
    %cd ..

os.chdir('T2I-RL-Project')
sys.path.insert(0, '.')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Verify imports
import torch
import transformers
from janus.models import VLChatProcessor, MultiModalityCausalLM

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Models

In [ ]:
import torch
import gc
from src.models.generators import JanusProGenerator, GenerationConfig
from src.models.reward_models import CLIPRewardModel

# Clear memory
gc.collect()
torch.cuda.empty_cache()

print("="*60)
print("Loading Janus-Pro-1B (bfloat16)")
print("="*60)

# Load generator - use bfloat16 for T4 with 16GB VRAM
generator = JanusProGenerator(
    model_name_or_path="deepseek-ai/Janus-Pro-1B",
    device="cuda",
    dtype=torch.bfloat16,
)
generator.load_model()

print(f"\nGPU Memory after generator: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("\n" + "="*60)
print("Loading CLIP Reward Model")
print("="*60)

# Load CLIP reward model
clip_reward = CLIPRewardModel(
    model_name="ViT-B-32",
    pretrained="openai",
    device="cuda",
)
clip_reward.load_model()

print(f"\nGPU Memory after CLIP: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("\nModels loaded successfully!")

## 3. Test Image Generation

In [ ]:
import matplotlib.pyplot as plt

# Test generation
test_prompts = [
    "a red apple on a wooden table",
    "a blue car parked next to a yellow house",
]

config = GenerationConfig(
    num_inference_steps=30,  # Reduce for faster testing
    guidance_scale=5.0,
    temperature=1.0,
    seed=42,
)

print("Generating test images...")
images = []
for prompt in test_prompts:
    img = generator.generate(prompt, config)
    images.extend(img)
    print(f"  Generated: {prompt[:40]}...")

# Display
fig, axes = plt.subplots(1, len(images), figsize=(6*len(images), 6))
for ax, img, prompt in zip(axes, images, test_prompts):
    ax.imshow(img)
    ax.set_title(prompt, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

# Test CLIP reward
reward_out = clip_reward.compute_reward(images, test_prompts)
print(f"\nCLIP Rewards: {reward_out.rewards.tolist()}")

## 4. Setup Training

In [ ]:
from torch.utils.data import DataLoader
from src.data.dataset import PromptDataset
from src.training.grpo_trainer import GRPOTrainer, GRPOConfig
import json

# Load training prompts
with open('data/prompts/train_prompts.json', 'r') as f:
    train_data = json.load(f)
    train_prompts = [p['prompt'] for p in train_data['prompts'][:50]]  # Start with 50 prompts

print(f"Training prompts: {len(train_prompts)}")
print(f"Examples: {train_prompts[:3]}")

# Create dataset and dataloader
train_dataset = PromptDataset(train_prompts)
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=2,  # Small batch for T4
    shuffle=True,
    num_workers=0,
)

# Training config optimized for T4 16GB
grpo_config = GRPOConfig(
    learning_rate=1e-5,
    num_epochs=1,
    batch_size=2,
    num_samples_per_prompt=2,  # Generate 2 samples per prompt for GRPO
    gradient_accumulation_steps=4,  # Effective batch size = 8
    kl_coef=0.1,
    clip_range=0.2,
    max_grad_norm=1.0,
    warmup_steps=10,
    use_wandb=False,  # Set True if you have wandb configured
    output_dir="./outputs/grpo_clip",
    save_steps=50,
    log_steps=10,
)

print(f"\nConfig:")
print(f"  Learning rate: {grpo_config.learning_rate}")
print(f"  Effective batch size: {grpo_config.batch_size * grpo_config.gradient_accumulation_steps}")
print(f"  Samples per prompt: {grpo_config.num_samples_per_prompt}")

In [ ]:
# Enable LoRA for efficient fine-tuning
print("Enabling LoRA...")
generator.enable_lora(lora_config={
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
})

# Count trainable parameters
trainable = sum(p.numel() for p in generator.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in generator.model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

print(f"\nGPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Create trainer
trainer = GRPOTrainer(
    generator=generator,
    reward_model=clip_reward,
    config=grpo_config,
    train_dataloader=train_dataloader,
)

print("Trainer created successfully!")

## 5. Run Training

In [ ]:
# Training loop
import time
from tqdm.auto import tqdm

print("="*60)
print("Starting GRPO Training")
print("="*60)

start_time = time.time()
step = 0
total_loss = 0
total_reward = 0

for epoch in range(grpo_config.num_epochs):
    print(f"\nEpoch {epoch + 1}/{grpo_config.num_epochs}")
    
    progress = tqdm(train_dataloader, desc="Training")
    for batch in progress:
        try:
            # Compute loss
            loss_dict = trainer.compute_loss(batch)
            loss = loss_dict['total_loss']
            
            # Backward pass
            loss.backward()
            
            # Gradient accumulation
            if (step + 1) % grpo_config.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(
                    generator.model.parameters(), 
                    grpo_config.max_grad_norm
                )
                trainer.optimizer.step()
                trainer.optimizer.zero_grad()
            
            # Logging
            total_loss += loss.item()
            if 'mean_reward' in loss_dict:
                total_reward += loss_dict['mean_reward']
            
            step += 1
            
            if step % grpo_config.log_steps == 0:
                avg_loss = total_loss / grpo_config.log_steps
                avg_reward = total_reward / grpo_config.log_steps
                progress.set_postfix({
                    'loss': f'{avg_loss:.4f}',
                    'reward': f'{avg_reward:.4f}',
                    'mem': f'{torch.cuda.memory_allocated()/1e9:.1f}GB'
                })
                total_loss = 0
                total_reward = 0
            
            # Save checkpoint
            if step % grpo_config.save_steps == 0:
                save_path = f"{grpo_config.output_dir}/checkpoint-{step}"
                generator.save_lora(save_path)
                print(f"\nSaved checkpoint to {save_path}")
                
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\nOOM at step {step}, clearing cache...")
                torch.cuda.empty_cache()
                gc.collect()
                continue
            raise e

elapsed = time.time() - start_time
print(f"\n" + "="*60)
print(f"Training complete!")
print(f"Total steps: {step}")
print(f"Time: {elapsed/60:.1f} minutes")
print("="*60)

## 6. Save Final Model

In [ ]:
# Save final LoRA weights
final_path = f"{grpo_config.output_dir}/final"
generator.save_lora(final_path)
print(f"Saved final model to {final_path}")

# Download to local
!zip -r outputs.zip outputs/
print("\nDownload outputs.zip to save your trained model!")

## 7. Evaluate Trained Model

In [ ]:
# Generate images with trained model
eval_prompts = [
    "a red apple on a wooden table",
    "a blue car parked next to a yellow house",
    "two cats playing with a red ball",
    "a chef cooking in a modern kitchen",
]

print("Generating evaluation images...\n")

eval_config = GenerationConfig(
    num_inference_steps=50,
    guidance_scale=5.0,
    temperature=1.0,
    seed=42,
)

eval_images = []
for prompt in eval_prompts:
    img = generator.generate(prompt, eval_config)
    eval_images.extend(img)

# Display
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, img, prompt in zip(axes.flat, eval_images, eval_prompts):
    ax.imshow(img)
    ax.set_title(prompt, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

# Compute rewards
eval_rewards = clip_reward.compute_reward(eval_images, eval_prompts)
print(f"\nEvaluation CLIP Rewards:")
for prompt, reward in zip(eval_prompts, eval_rewards.rewards):
    print(f"  {prompt[:40]}... : {reward.item():.4f}")
print(f"\nMean Reward: {eval_rewards.rewards.mean().item():.4f}")

## 8. GPU Memory Report

In [ ]:
print("="*60)
print("GPU Memory Report")
print("="*60)

allocated = torch.cuda.memory_allocated() / 1e9
max_allocated = torch.cuda.max_memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"Current: {allocated:.2f} GB")
print(f"Peak:    {max_allocated:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")
print(f"Total:   {total:.2f} GB")
print(f"\nUtilization: {100*max_allocated/total:.1f}%")

---
## Done!

Your trained LoRA weights are saved in `outputs/grpo_clip/final/`.

To use them later:
```python
generator.load_lora("path/to/lora/weights")
```